In [10]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import torch

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.model.ensemble_model import EnsembleModel
from src.model.game_model_collection import GamingModelCollection
from src.model.social_media_collection import SocialMediaModelCollection
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

In [11]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

In [12]:
# social media
social_binary_paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))
scaler_binary_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_binary_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'

social_multi_paths = list((CONFIG['model_dir'] / 'multi' ).glob('social_media_*.joblib'))
scaler_multi_path = CONFIG['model_dir'] / 'helper' / 'multi_max_abs_scaler.joblib'
nb_tfidf_multi_path = CONFIG['model_dir'] / 'helper' / 'multi_word_tfidf_vectorizer.joblib'

# gaming
gaming_binary_dir = CONFIG['model_dir'] / "binary"
gaming_binary_model_paths = list(gaming_binary_dir.glob("gaming_all_*.joblib"))
gaming_multi_dir = CONFIG['model_dir'] / "multi"
gaming_multi_model_paths = list(gaming_multi_dir.glob("gaming_all_*.joblib"))

# Load Data

In [13]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [14]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# Prepare Objects


In [15]:
# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

gaming_preprocessor.fit(train, text_column=tc)

In [ ]:
# create 3-class
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)

# create binary
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary',
)

val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary',
)


In [ ]:
# create gaming multi collection 

gaming_multi_model_collection = GamingModelCollection(
    model_joblibs = gaming_multi_model_paths
)

gaming_multi_weights = [0.01500173, 0.06092781, 0.00230242, 0.04228979, 0.04470104,
        0.05293533, 0.09912261, 0.00050821, 0.35757238, 0.06847398,
        0.22296926, 0.03319543]

gaming_multi_threshold = 0.7000000000000002


# create gaming binary collection

gaming_binary_model_collection = GamingModelCollection(
    model_joblibs = gaming_binary_model_paths
)

gaming_binary_weights = [0.04523348, 0.00946077, 0.08420375, 0.03204597, 0.09418352,
        0.03581342, 0.00528644, 0.04582971, 0.49255951, 0.02053279,
        0.05121582, 0.08363481]

gaming_binary_threshold = 0.8000000000000003


In [ ]:
# create social media multi collection
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_multi_paths,
    scaler_path=scaler_multi_path,
    nb_tfidf_path=nb_tfidf_multi_path,
)


# create social media binary collection
social_media_binary_collection = SocialMediaModelCollection(
    model_joblibs=social_binary_paths,
    scaler_path=scaler_binary_path,
    nb_tfidf_path=nb_tfidf_binary_path,
)

social_media_binary_weights = [0.45, 0.55, 0.00]
social_media_binary_threshold = 0.74

In [19]:
# create bert multi collection
bert_multi_collection = BertToxicityModelCollection(
    model_names=["dota_multi", "jigsaw_multi", "wot_multi"]
)

# create bert binary collection
bert_binary_collection = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_multi from jforward/bert-toxicity/dota_multi_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6416.34it/s]


Loading jigsaw_multi from jforward/bert-toxicity/jigsaw_multi_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5655.70it/s]


Loading wot_multi from jforward/bert-toxicity/wot_multi_model...


...

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7025.99it/s]


Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6942.50it/s]


Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6131.22it/s]
